# Práctica de la lezione 12: mini-proyecto de cierre con el dataset de vinos

Esta es la última pratica del curso, y funciona como un mini-proyecto de cierre en dos pasos. Trabajas sobre `load_wine()`, el dataset de vinos que ya usaste en praticas anteriores, con trece feature químicas (grado alcohólico, acidez, magnesio, color, entre otras) y tres leziones de vino ya conocidas.

En el **paso uno** eliges tú, entre K vecinos más próximos (KNN, del inglés *K-Nearest Neighbors*) y máquinas de vectores de soporte (SVM, del inglés *Support Vector Machines*), qué algoritmo usar para clasificar la lezione de vino, y calculas su score. En el **paso dos** olvidas la colonna de lezione y aplicas K-Means para ver si el propio dataset, sin ninguna etiqueta, se agrupa de forma parecida a como lo haría un enólogo o una enóloga con las tres variedades reales.

Cada paso trae un consegna con suggerimenti, una cella de código con partes marcadas como `# TODO` para que completes, y más abajo una soluzione colapsada por si te atascas. Intenta resolverlo da solo antes de mirarla: es la última oportunidad del curso para comprobar cuánto has aprendido.

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

vino = load_wine()
X = vino.data
y = vino.target

print("Forma de X:", X.shape)
print("Clases:", vino.target_names)
print("Muestras por clase:", pd.Series(y).value_counts().sort_index().tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Paso 1: clasifica la lezione de vino con KNN o con SVM

Ya tienes `X_train`, `X_test`, `y_train`, `y_test` preparados en la cella anterior. Elige un algoritmo de clasificación de los que viste en el curso, `KNeighborsClassifier` o `SVC`, entrénalo con los dati de addestramento y calcula su score sobre el conjunto de prueba.

**Suggerimenti:**

- Si eliges KNN, impórtalo con `from sklearn.neighbors import KNeighborsClassifier` y crea el modello con `KNeighborsClassifier(n_neighbors=5)`, igual que en las leziones cinco y seis
- Si eliges SVM, impórtalo con `from sklearn.svm import SVC` y crea el modello con `SVC()`, igual que en las leziones siete a nueve
- Entrena el modello con `.fit(X_train, y_train)`
- El score se calcula con `.score(X_test, y_test)` sobre el modello ya entrenado
- No hace falta que pruebes los dos: elige uno solo y quédate con su score. Si tienes curiosidad, prueba el otro después de ver la soluzione y compara

In [ ]:
# TODO 1: importa el algoritmo que elijas, KNeighborsClassifier o SVC

# TODO 2: crea el modelo (por ejemplo, KNeighborsClassifier(n_neighbors=5) o SVC())
modelo_clasificacion = ...

# TODO 3: entrena el modelo con los datos de entrenamiento
...

# TODO 4: calcula el score del modelo sobre el conjunto de prueba
score_clasificacion = ...

print("Score del modelo de clasificación:", score_clasificacion)

<details>
<summary>💡 Vedi soluzione</summary>

```python
# Esta soluzione usa KNN a modo de ejemplo; con SVC el planteamiento es idéntico,
# solo cambia el import y la línea donde se crea el modello

# TODO 1: importamos el algoritmo elegido
from sklearn.neighbors import KNeighborsClassifier

# TODO 2: creamos el modello
modello_clasificacion = KNeighborsClassifier(n_neighbors=5)

# TODO 3: entrenamos el modello con los dati de addestramento
modello_clasificacion.fit(X_train, y_train)

# TODO 4: calculamos el score sobre el conjunto de prueba
score_clasificacion = modello_clasificacion.score(X_test, y_test)

print("Score del modello de clasificación:", score_clasificacion)

# Alternativa con SVM, por si quieres comparar:
# from sklearn.svm import SVC
# modello_clasificacion = SVC()
# modello_clasificacion.fit(X_train, y_train)
# score_clasificacion = modello_clasificacion.score(X_test, y_test)
```

Con KNN y `n_neighbors=5`, el score sobre el conjunto de prueba ronda 0,806. Con SVC y su configuración por defecto, ronda 0,694, más bajo en este dataset concreto porque las trece feature del vino tienen escalas muy distintas entre sí (el grado alcohólico se mueve en un rango pequeño, mientras que la prolina llega a cientos), y SVC es más sensible a esa diferencia de escala que KNN. Si quieres mejorar el risultato de SVC, prueba a escalar las feature antes de addestrare con `StandardScaler`, la misma herramienta que ya viste en leziones anteriores.

</details>

## Paso 2: agrupa el dataset con K-Means, sin usar la lezione

Ahora olvida que conoces la lezione de cada vino. Aplica `KMeans` directamente sobre `X` (las trece feature, sin la colonna de lezione) pidiendo tres clusters, ya que sabes de antemano que hay tres variedades reales, y compara los grupos que encuentra contra las leziones reales guardadas en `y`.

**Suggerimenti:**

- Importa `KMeans` con `from sklearn.cluster import KMeans`
- Crea el modello con `KMeans(n_clusters=3, random_state=42, n_init=10)`; fija `random_state` para que el risultato sea reproducible
- Agrupa y obtén las etiquetas de cluster en un solo paso con `.fit_predict(X)`
- Para comparar los grupos encontrados contra las leziones reales, usa `pd.crosstab(grupos_encontrados, y)`, la misma tabla cruzada que viste en lezione; cada riga es un cluster, cada colonna una lezione real, y los números indican cuántos vinos de esa lezione cayeron en ese cluster

In [ ]:
from sklearn.cluster import KMeans

# TODO 1: crea un modelo KMeans con 3 clusters y random_state=42, n_init=10
modelo_clustering = ...

# TODO 2: agrupa las muestras de X y guarda las etiquetas de cluster
grupos_encontrados = ...

# TODO 3: construye la tabla cruzada entre grupos_encontrados y las clases reales y
tabla_comparacion = ...

tabla_comparacion

<details>
<summary>💡 Vedi soluzione</summary>

```python
from sklearn.cluster import KMeans

# TODO 1: creamos el modello KMeans con 3 clusters
modello_clustering = KMeans(n_clusters=3, random_state=42, n_init=10)

# TODO 2: agrupamos las muestras de X y guardamos las etiquetas de cluster
grupos_encontrados = modello_clustering.fit_predict(X)

# TODO 3: comparamos los grupos encontrados con las leziones reales
tabla_comparacion = pd.crosstab(grupos_encontrados, y,
                                 rownames=["Cluster de KMeans"],
                                 colnames=["Clase real"])

tabla_comparacion
```

Al ejecutarlo ves tres clusters que no coinciden perfectamente con las tres leziones reales, pero tampoco al azar: cada cluster concentra la mayoría de sus vinos en una o dos leziones concretas, en lugar de repartirlos por igual entre las tres. Por ejemplo, uno de los clusters agrupa casi en exclusiva vinos de una sola lezione real, mientras que otro mezcla vinos de dos leziones distintas. Esa mezcla parcial tiene sentido: K-Means solo ve las trece feature químicas, sin ninguna etiqueta, y algunas variedades de vino se parecen más entre sí en esas medidas que otras. Aun así, que los grupos se acerquen bastante a las leziones reales confirma la misma idea que viste con el cáncer de mama en el notebook de lezione: la estructura de los dati ya contiene buena parte de la información que necesitas, incluso antes de mirar ninguna etiqueta.

</details>

## Resumen: lo que te llevas de esta pratica

- Con dati ya etiquetados, como la lezione de vino, tanto KNN como SVC son opciones válidas de clasificación; su rendimiento puede variar bastante según cómo estén escaladas las feature de tu dataset
- K-Means, aplicado sobre las mismas feature pero sin la colonna de lezione, encuentra grupos que se parecen —aunque no coinciden a la perfección— con las leziones reales del dataset
- Comparar los clusters de un algoritmo no supervisado contra una etiqueta real que sí conoces, cuando la tienes disponible, es una forma habitual de comprobar si el clustering está capturando algo con sentido
- Con este mini-proyecto cierras el recorrido completo del curso: regresión, KNN, SVM, K-Means y Mean Shift, los cinco aplicados sobre datasets reales

Enhorabuena por llegar hasta aquí. Ya tienes la caja de herramientas completa; el siguiente paso es abrirla sobre un problema propio.

**Abbreviazioni usate en esta pratica:** KNN (*K-Nearest Neighbors*, K vecinos más próximos), SVM (*Support Vector Machines*, máquinas de vectores de soporte).